# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library. All references to dataset entities use their Croissant `@id` properties, ensuring clarity and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (uncomment below line if not already installed)
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and list available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print('Name     :', meta.name)
print('Version  :', meta.version)
print('Identifier:', meta.identifier)
print('Desc     :', meta.description)


## 2. Data Overview
List available record sets, fields, and their `@id`s. `mlcroissant` provides programmatic access to examine the data structure as defined in the Croissant schema.

**Note:** Entities in Croissant schemas are uniquely referenced by their `@id`. Always reference record sets, fields, and columns by their `@id` for consistency.

In [ ]:
# List all record sets and their available fields
record_sets = dataset.record_sets  # This is a dict: {record_set_id: RecordSet, ...}
print('Available record sets and fields:')
for rs_id, rs in record_sets.items():
    print(f'  RecordSet @id: {rs_id}')
    print(f'    name: {getattr(rs, "name", "(none)")}', )
    print(f'    description: {getattr(rs, "description", "(none)")}', )
    print(f'    Fields:')
    for field in rs.fields:
        print(f'      Field @id: {field.id}')
        print(f'        name: {getattr(field, "name", "(none)")}', )
        print(f'        Data type: {getattr(field, "data_type", "(unknown)")}', )
    print()

## 3. Data Extraction
Load records from a selected record set into a DataFrame. All references use Croissant `@id` (not just the display `name`).

Replace the variables below as needed based on the above overview.

In [ ]:
# List of record set `@id`s (from previous cell's output)
record_set_ids = list(record_sets.keys())
print('Record sets in this dataset:', record_set_ids)

# For demonstration, load all record sets into separate DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading records from record set: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))  # each record is a dict with field @id as the key
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display the available columns (which are field @id values) for the first record set as an example
if record_set_ids:
    example_rs = record_set_ids[0]
    print(f'Columns (field @id) in record set {example_rs}:')
    print(dataframes[example_rs].columns.values.tolist())
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
We will demonstrate basic numeric filtering, normalization, and grouping using the first numeric field (referenced by its `@id`).

Replace the field IDs below as needed. All `@id`s must be used directly for proper data access.

In [ ]:
import numpy as np
# Identify the first numeric field (by @id) in the selected record set
target_record_set = example_rs
numeric_field_id = None

for field in record_sets[target_record_set].fields:
    if getattr(field, 'data_type', None) in ['Integer', 'Float', 'Number']:
        numeric_field_id = field.id
        print('Selected numeric field for EDA (by @id):', numeric_field_id)
        break

df = dataframes[target_record_set].copy()

# Ensure numeric conversion (if field exists and not all missing)
if numeric_field_id and numeric_field_id in df:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Example: Filter for values greater than threshold
    threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].notna().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f'Filtered records where {numeric_field_id} > {threshold}')
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by a categorical field (if available)
    group_field_id = None
    for field in record_sets[target_record_set].fields:
        if getattr(field, 'data_type', None) == 'Text' and field.id != numeric_field_id:
            group_field_id = field.id
            break
    if group_field_id and group_field_id in filtered_df:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped.head())
else:
    print('No numeric field detected or numeric field missing in records for EDA.')

## 5. Visualization
Create simple visualizations to explore data distributions or relationships. All axes are labeled by field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # If a group field is set, plot boxplot
    if group_field_id and group_field_id in df:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to:
- Load a Croissant dataset (referencing all entities by their `@id`).
- Survey its record sets and fields.
- Extract tabular data directly from the Croissant schema.
- Perform simple EDA: numeric filtering, normalization, grouping.
- Visualize field distributions and relationships.

This approach ensures strong adherence to the standards of the Croissant data model and fair, reproducible analysis workflows.